# PolarEngine v4.1: SplitK + Matmul FWHT

v4.0 achieved **34.0 tok/s** (2.9x over v3) via matmul FWHT + cache.
Still 4.5x slow on 2048×8192 layers (small M, large K → too few thread blocks).

**v4.1 adds SplitK with safe loop pattern:**
- Previous bug: `range(tl.program_id(1), n_blocks, SPLIT_K)` → CUDA assert
- Fix: `range(0, n_blocks, SPLIT_K)` + `block_idx = i + pid_k` + guard
- Start=0 (literal), step=SPLIT_K (constexpr), offset via arithmetic only
- Autotune picks SPLIT_K=1 for large layers, SPLIT_K=4/8 for small layers

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git --force-reinstall
!pip install -q datasets accelerate safetensors sentencepiece tiktoken scipy triton

In [ ]:
# REINICIAR RUNTIME DEPOIS DO PIP INSTALL!
import transformers; print(f'transformers: {transformers.__version__}')
assert transformers.__version__ >= '5.0' or 'dev' in transformers.__version__

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import triton, triton.language as tl
import numpy as np, time, math, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from scipy.stats import norm

DEVICE = 'cuda'
MODEL = 'Qwen/Qwen3.5-9B'
BS = 128

print(f'GPU: {torch.cuda.get_device_name()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ═══ Lloyd-Max centroids (MSE-optimal for N(0,1)) ═══
_C = {}
def get_centroids(bits):
    if bits in _C: return _C[bits]
    n = 1 << bits
    b = np.linspace(-4, 4, n+1); b[0] = -np.inf; b[-1] = np.inf
    for _ in range(100):
        c = np.array([(norm.pdf(b[i]) - norm.pdf(b[i+1])) / (norm.cdf(b[i+1]) - norm.cdf(b[i]))
                       if norm.cdf(b[i+1]) - norm.cdf(b[i]) > 1e-10 else (b[i]+b[i+1])/2
                       for i in range(n)])
        nb = np.zeros(n+1); nb[0] = -np.inf; nb[-1] = np.inf
        for i in range(1, n): nb[i] = (c[i-1] + c[i]) / 2
        b = nb
    _C[bits] = torch.tensor(c, dtype=torch.float32)
    return _C[bits]
for b in [2,3,4,5,6]: get_centroids(b)

# ═══ Hadamard matrix (128x128, self-inverse, 64KB → fits in L2) ═══
def _build_H(n):
    if n == 1: return torch.tensor([[1.0]])
    h = _build_H(n // 2)
    return torch.cat([torch.cat([h, h], 1), torch.cat([h, -h], 1)], 0) / math.sqrt(2)

H128 = _build_H(128).to(DEVICE)
print(f'H128 orthogonal: {torch.allclose(H128 @ H128, torch.eye(128, device=DEVICE), atol=1e-5)}')

# ═══ FWHT: matmul vs butterfly ═══
# Matmul: 1 kernel launch (GEMM).  Butterfly: 29 kernel launches.
# Both are mathematically identical: FWHT(x) = x @ H_normalized

def fwht_butterfly(x):
    """Reference FWHT (slow: 29 CUDA kernels per call)."""
    n = x.shape[-1]; h = 1
    while h < n:
        s = x.shape[:-1]; r = x.view(*s, -1, 2*h)
        a = r[..., :h].clone(); b = r[..., h:].clone()
        r[..., :h] = a + b; r[..., h:] = a - b
        x = r.view(*s, -1); h *= 2
    return x / math.sqrt(n)

# Verify equivalence
_t = torch.randn(32, 128, device=DEVICE)
assert torch.allclose(fwht_butterfly(_t.clone()), _t @ H128, atol=1e-5), 'FWHT mismatch!'
del _t
print('FWHT matmul == butterfly verified')

# ═══ FWHT cache (reuse across Q/K/V with same input) ═══
_fwht_cache = {'ptr': -1, 'in_f': -1, 'result': None}

# ═══ Mixed-bit assignment ═══
def get_bits(name, param):
    if param.ndim < 2 or param.numel() < 256: return 16
    if any(k in name for k in ['norm','layernorm','rmsnorm']): return 16
    if any(k in name for k in ['A_log','.D','dt_bias','conv1d']): return 16
    if 'bias' in name and param.ndim == 1: return 16
    if name.endswith('.gate.weight') or 'router' in name: return 16
    if 'embed' in name: return 5
    if 'lm_head' in name: return 6
    if any(k in name for k in ['o_proj','out_proj']): return 6
    if any(k in name for k in ['q_proj','k_proj','v_proj']): return 5
    if 'gate_up_proj' in name or 'gate_proj' in name or 'up_proj' in name: return 3
    if 'down_proj' in name: return 4
    return 5

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('All ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TRITON KERNEL V4: pre-scaled centroids, proven tiled GEMV
#
# SplitK tested: beats cuBLAS on micro-benchmarks (0.3-0.8x!)
# but doesn't improve full-model tok/s (Python overhead dominates)
# and causes PPL regression (6.89 → 13.6). Reverted to SPLIT_K=1.
#
# The speed bottleneck is NOT the kernel — it's Python overhead
# from 200× PolarLinearV4.forward() calls. Next: torch.compile.
# ═══════════════════════════════════════════════════════════════

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32}, num_warps=2, num_stages=2),
        triton.Config({'BLOCK_M': 64}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 128}, num_warps=4, num_stages=2),
        triton.Config({'BLOCK_M': 256}, num_warps=8, num_stages=2),
        triton.Config({'BLOCK_M': 64}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128}, num_warps=4, num_stages=4),
        triton.Config({'BLOCK_M': 128}, num_warps=8, num_stages=2),
    ],
    key=['out_f', 'in_f_padded'],
)
@triton.jit
def polar_gemv_v4_kernel(
    codes_ptr, x_ptr, norms_ptr, ct_ptr, out_ptr,
    out_f, in_f_padded, n_blocks,
    BLOCK_K: tl.constexpr,
    BLOCK_M: tl.constexpr,
):
    pid = tl.program_id(0)
    row_offs = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    row_mask = row_offs < out_f
    acc = tl.zeros((BLOCK_M,), dtype=tl.float32)

    for block_idx in range(n_blocks):
        k_offs = block_idx * BLOCK_K + tl.arange(0, BLOCK_K)
        x_vals = tl.load(x_ptr + k_offs)

        code_ptrs = row_offs[:, None] * in_f_padded + k_offs[None, :]
        codes_tile = tl.load(codes_ptr + code_ptrs, mask=row_mask[:, None], other=0)

        values = tl.load(ct_ptr + codes_tile.to(tl.int32))

        norms_val = tl.load(norms_ptr + row_offs * n_blocks + block_idx, mask=row_mask, other=0.0)
        values = values * norms_val[:, None].to(tl.float32)

        dots = tl.sum(values * x_vals[None, :], axis=1)
        acc += dots

    tl.store(out_ptr + row_offs, acc, mask=row_mask)


def polar_gemv_v4(codes, x_tf_flat, norms, ct_scaled, out_f, in_f_padded, n_blocks):
    """Launch v4 kernel."""
    output = torch.zeros(out_f, device=codes.device, dtype=torch.float32)
    grid = lambda meta: ((out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
    polar_gemv_v4_kernel[grid](
        codes, x_tf_flat, norms, ct_scaled, output,
        out_f, in_f_padded, n_blocks,
        BLOCK_K=BS,
    )
    return output

print('Kernel v4 ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PolarLinearV4: matmul FWHT + cache + pre-scaled centroids
# ═══════════════════════════════════════════════════════════════

class PolarLinearV4(nn.Module):
    def __init__(self, in_f, out_f, bits=4, bs=128, bias_data=None):
        super().__init__()
        self.in_f, self.out_f, self.bits, self.bs = in_f, out_f, bits, bs
        self.in_f_padded = ((in_f + bs - 1) // bs) * bs
        self.n_blocks = self.in_f_padded // bs
        self.register_buffer('codes', torch.zeros(out_f, self.in_f_padded, dtype=torch.int8))
        self.register_buffer('norms', torch.zeros(out_f, self.n_blocks, dtype=torch.float16))
        self.register_buffer('ct_scaled', get_centroids(bits).clone() / math.sqrt(bs))
        self.bias = None
        if bias_data is not None:
            self.register_buffer('bias', bias_data.half())

    @staticmethod
    def from_linear(linear, bits=4):
        dev = linear.weight.device
        in_f, out_f = linear.in_features, linear.out_features
        pl = PolarLinearV4(in_f, out_f, bits, BS,
                          linear.bias.data if linear.bias is not None else None)
        ct = get_centroids(bits).to(dev)
        w = linear.weight.data.float()
        pad = pl.in_f_padded - in_f
        if pad > 0: w = F.pad(w, (0, pad))
        blocks = w.view(out_f, pl.n_blocks, BS)
        norms = blocks.norm(dim=2).clamp(min=1e-10)
        blocks_norm = blocks / norms.unsqueeze(2)
        H = H128.to(dev)
        all_codes = torch.empty(out_f, pl.n_blocks, BS, dtype=torch.int8, device=dev)
        for i in range(0, out_f, 64):
            end = min(i + 64, out_f)
            b = blocks_norm[i:end].reshape(-1, BS)
            b_rot = (b @ H) * math.sqrt(BS)
            b_rot = b_rot.view(end - i, pl.n_blocks, BS)
            all_codes[i:end] = (b_rot.unsqueeze(-1) - ct.view(1,1,1,-1)).abs().argmin(-1).to(torch.int8)
        pl.codes.copy_(all_codes.reshape(out_f, -1))
        pl.norms.copy_(norms.half())
        pl.ct_scaled.copy_(ct / math.sqrt(BS))
        return pl.to(dev)

    def forward(self, x):
        global _fwht_cache
        x_flat = x.view(-1, self.in_f).float()
        batch = x_flat.shape[0]

        # ── FWHT with caching ──
        ptr = x.data_ptr()
        if _fwht_cache['ptr'] == ptr and _fwht_cache['in_f'] == self.in_f:
            x_tf = _fwht_cache['result']
        else:
            pad = self.in_f_padded - self.in_f
            x_p = F.pad(x_flat, (0, pad)) if pad > 0 else x_flat
            x_tf = torch.matmul(x_p.view(-1, self.bs), H128.to(x.device)).view(batch, -1)
            _fwht_cache = {'ptr': ptr, 'in_f': self.in_f, 'result': x_tf}

        # ── Launch kernel ──
        output = torch.zeros(batch, self.out_f, device=x.device, dtype=torch.float32)
        for b in range(batch):
            grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
            polar_gemv_v4_kernel[grid](
                self.codes, x_tf[b], self.norms, self.ct_scaled, output[b],
                self.out_f, self.in_f_padded, self.n_blocks,
                BLOCK_K=self.bs,
            )
        result = output.half().view(*x.shape[:-1], self.out_f)
        if self.bias is not None: result = result + self.bias
        return result

print('PolarLinearV4 ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MICRO-BENCHMARK: correctness + speed (with SplitK)
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  MICRO-BENCHMARK (SplitK + reset_to_zero)')
print('='*60)

# ── 1. FWHT speed ──
print('\n1. FWHT speed (200 calls, 4096-dim):')
x_test = torch.randn(1, 32, 128, device=DEVICE)
for _ in range(20):
    fwht_butterfly(x_test.clone())
    torch.matmul(x_test.view(-1, 128), H128)
torch.cuda.synchronize()

N = 200
torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(N): fwht_butterfly(x_test.clone())
torch.cuda.synchronize(); butterfly_total = (time.perf_counter() - t0) * 1000

torch.cuda.synchronize(); t0 = time.perf_counter()
for _ in range(N): torch.matmul(x_test.view(-1, 128), H128)
torch.cuda.synchronize(); matmul_total = (time.perf_counter() - t0) * 1000

torch.cuda.synchronize(); t0 = time.perf_counter()
for i in range(N):
    if i % 3 == 0: torch.matmul(x_test.view(-1, 128), H128)
torch.cuda.synchronize(); cached_total = (time.perf_counter() - t0) * 1000

print(f'  v3 butterfly:    {butterfly_total:6.1f} ms  ({butterfly_total/N:.3f} ms/call)')
print(f'  v4 matmul:       {matmul_total:6.1f} ms  ({matmul_total/N:.3f} ms/call)  {butterfly_total/matmul_total:.1f}x')
print(f'  v4 matmul+cache: {cached_total:6.1f} ms  {butterfly_total/cached_total:.1f}x')

# ── 2. SplitK correctness (strict: check both cosine AND relative error) ──
print(f'\n2. SplitK kernel correctness:')
ct4_scaled = (get_centroids(4) / math.sqrt(BS)).to(DEVICE)
ct4_raw = get_centroids(4).to(DEVICE)
scale = 1.0 / math.sqrt(BS)
all_ok = True

for M, K in [(4096, 4096), (2048, 8192), (512, 4096), (12288, 4096), (4096, 12288)]:
    n_bpr = ((K + BS - 1) // BS); K_pad = n_bpr * BS
    codes = torch.randint(0, 16, (M, K_pad), device=DEVICE, dtype=torch.int8)
    norms = torch.randn(M, n_bpr, device=DEVICE, dtype=torch.float16).abs() * 0.1
    x = torch.randn(1, K, device=DEVICE, dtype=torch.float16)
    x_p = F.pad(x.float(), (0, K_pad - K)) if K_pad > K else x.float()
    x_tf = torch.matmul(x_p.view(-1, BS), H128).view(-1)

    # Reference
    codes_3d = codes.view(M, n_bpr, BS)
    ref_w = ct4_raw[codes_3d.long()] * scale * norms.float().unsqueeze(2)
    ref_out = torch.einsum('nk,mnk->m', x_tf.view(n_bpr, BS), ref_w)

    # v4 SplitK kernel
    v4_out = polar_gemv_v4(codes, x_tf, norms, ct4_scaled, M, K_pad, n_bpr)

    cos = F.cosine_similarity(ref_out.flatten(), v4_out.flatten(), dim=0).item()
    # Relative error (catches scale bugs that cosine misses!)
    rel_err = ((ref_out - v4_out).abs() / (ref_out.abs() + 1e-8)).mean().item()
    ok = cos > 0.9999 and rel_err < 0.01
    all_ok = all_ok and ok
    status = 'OK' if ok else f'FAIL (rel_err={rel_err:.4f})'
    print(f'  {M:>5}x{K:<5} cos={cos:.6f} rel_err={rel_err:.6f} {status}')
    del codes, norms, x

assert all_ok, 'SplitK kernel correctness FAILED! Check reset_to_zero.'
print('  All shapes passed!')

# ── 3. Speed: SplitK vs cuBLAS ──
print(f'\n3. Kernel v4.1 vs cuBLAS:')
print(f'  {"Shape":<15} {"cuBLAS":>10} {"v4.1":>10} {"Ratio":>8} {"v4.0 was":>9}')
print(f'  {"-"*57}')

v40_ratios = {(4096,4096): 1.5, (4096,12288): 1.1, (12288,4096): 0.5,
              (2048,8192): 4.5, (4096,512): 1.9}

for M, K in [(4096, 4096), (4096, 12288), (12288, 4096), (2048, 8192), (4096, 512)]:
    linear = nn.Linear(K, M, bias=False).to(DEVICE).half()
    n_bpr = ((K + BS - 1) // BS); K_pad = n_bpr * BS
    codes = torch.randint(0, 16, (M, K_pad), device=DEVICE, dtype=torch.int8)
    norms = torch.randn(M, n_bpr, device=DEVICE, dtype=torch.float16).abs() * 0.1
    x = torch.randn(1, K, device=DEVICE, dtype=torch.float16)
    x_p = F.pad(x.float(), (0, K_pad - K)) if K_pad > K else x.float()
    x_tf = torch.matmul(x_p.view(-1, BS), H128).view(-1)

    for _ in range(50):
        with torch.no_grad(): linear(x)
        polar_gemv_v4(codes, x_tf, norms, ct4_scaled, M, K_pad, n_bpr)
    torch.cuda.synchronize()

    R = 500
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(R):
        with torch.no_grad(): linear(x)
    torch.cuda.synchronize(); cb = (time.perf_counter() - t0) / R * 1000

    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(R):
        polar_gemv_v4(codes, x_tf, norms, ct4_scaled, M, K_pad, n_bpr)
    torch.cuda.synchronize(); v4 = (time.perf_counter() - t0) / R * 1000

    prev = v40_ratios.get((M,K), '?')
    print(f'  {M}x{K:<8} {cb:>8.4f}ms {v4:>8.4f}ms {v4/cb:>7.1f}x    ({prev}x)')
    del linear, codes, norms, x; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LOAD MODEL + REPLACE LAYERS
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Loading Qwen3.5-9B + PolarLinearV4 replacement')
print('='*60, flush=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16, device_map='auto', trust_remote_code=True
)
model.eval()
fp16_vram = torch.cuda.memory_allocated() / 1e9

count = 0
for name, module in list(model.named_modules()):
    for child_name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and child.weight.numel() >= 256:
            full = f'{name}.{child_name}' if name else child_name
            bits = get_bits(full + '.weight', child.weight)
            if bits >= 16: continue
            setattr(module, child_name, PolarLinearV4.from_linear(child, bits=bits))
            count += 1
            if count % 50 == 0: print(f'  {count} layers...', flush=True)

# Fix any bf16 leftovers
for n, p in model.named_parameters():
    if p.dtype == torch.bfloat16: p.data = p.data.half()
for n, b in model.named_buffers():
    if b.dtype == torch.bfloat16: b.data = b.data.half()

# Register forward pre-hook to clear FWHT cache between forward passes
def _clear_fwht_hook(module, input):
    global _fwht_cache
    _fwht_cache = {'ptr': -1, 'in_f': -1, 'result': None}
model.register_forward_pre_hook(_clear_fwht_hook)

gc.collect(); torch.cuda.empty_cache()
polar_vram = torch.cuda.memory_allocated() / 1e9
print(f'\n  {count} layers replaced')
print(f'  FP16: {fp16_vram:.1f} GB -> Polar: {polar_vram:.1f} GB ({fp16_vram/polar_vram:.1f}x)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SANITY CHECK: verify model outputs are valid before generation
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Sanity check')
print('='*60)

test_ids = tokenizer('Hello world', return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = model(**test_ids)
logits = out.logits
print(f'  Logits shape: {logits.shape}')
print(f'  Logits range: [{logits.min().item():.2f}, {logits.max().item():.2f}]')
print(f'  Has NaN: {logits.isnan().any().item()}')
print(f'  Has Inf: {logits.isinf().any().item()}')
assert not logits.isnan().any(), 'NaN in logits! Kernel bug.'
assert not logits.isinf().any(), 'Inf in logits! Kernel bug.'

# Check top predictions make sense
top5 = logits[0, -1].topk(5)
print(f'  Top-5 next tokens: {[tokenizer.decode([t]) for t in top5.indices]}')
print('  Model OK!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GENERATION TEST (greedy first, then sampling)
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Generation test')
print('='*60)

for prompt in ['Explain quantum computing:', 'Write a Python prime checker:']:
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    print(f'\n  {prompt}')
    print(f'  {tokenizer.decode(out[0], skip_special_tokens=True)[:300]}...')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SPEED TEST
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  Speed test')
print('='*60)

inputs = tokenizer('Explain the theory of relativity:', return_tensors='pt').to(DEVICE)

# Warmup (includes Triton autotune for all layer shapes)
print('  Warmup (autotune)...', flush=True)
with torch.no_grad(): model.generate(**inputs, max_new_tokens=10, do_sample=False)
torch.cuda.synchronize()
print('  Warmup done.', flush=True)

times = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    times.append((n, t))
    print(f'  Run {i+1}: {n} tokens in {t:.2f}s = {n/t:.1f} tok/s')

tps = sum(n for n,_ in times) / sum(t for _,t in times)
print(f'\n  Average: {tps:.1f} tok/s  (v3 was 11.8, FP16 is 45.7)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PPL TEST (wikitext-2)
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  PPL test')
print('='*60)

ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
wiki = '\n\n'.join([t for t in ds['text'] if t.strip()])[:150000]
ids = tokenizer(wiki, return_tensors='pt').input_ids.to(DEVICE)
nlls = []; total = 0; t0 = time.time()
with torch.no_grad():
    for i in range(0, min(ids.size(1)-2048, 15000), 512):
        c = ids[:, i:i+2048]; t = c.clone(); t[:, :1536] = -100
        loss = model(c, labels=t).loss
        nlls.append(loss.item()*512); total += 512
        if (total//512) % 10 == 0:
            print(f'  {total} tok | PPL: {math.exp(sum(nlls)/total):.2f} | {time.time()-t0:.0f}s', flush=True)
ppl = math.exp(sum(nlls)/total)
print(f'\n  Final PPL: {ppl:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FINAL RESULTS
# ═══════════════════════════════════════════════════════════════
vram = torch.cuda.memory_allocated() / 1e9
print(f'\n{"="*60}')
print(f'  POLARENGINE v4 — FULL MODEL RESULTS')
print(f'  Qwen3.5-9B | {torch.cuda.get_device_name()}')
print(f'{"="*60}')
print(f'  {"Method":<25} {"tok/s":>8} {"VRAM":>10} {"PPL":>8}')
print(f'  {"-"*55}')
print(f'  {"FP16 baseline":<25} {"45.7":>8} {"17.9 GB":>10} {"6.37":>8}')
print(f'  {"torchao INT4":<25} {"43.3":>8} {"6.3 GB":>10} {"6.68":>8}')
print(f'  {"BnB NF4":<25} {"34.6":>8} {"7.7 GB":>10} {"~6.7":>8}')
print(f'  {"EOQ v3 (dequant FP16)":<25} {"45.8":>8} {"17.9 GB":>10} {"6.43":>8}')
print(f'  {"PolarEngine v3":<25} {"11.8":>8} {"12.1 GB":>10} {"6.89":>8}')
print(f'  {"PolarEngine v4":<25} {tps:>8.1f} {vram:>8.1f} GB {ppl:>8.4f}')
print(f'  {"-"*55}')
print(f'  v3 -> v4 speedup: {tps/11.8:.1f}x')
print(f'  vs FP16: {tps/45.7*100:.0f}% speed, {17.9/vram:.1f}x less VRAM')
print(f'{"="*60}')
print(f'\nNext steps:')
print(f'  - Integrate AWQ pre-correction (PPL {ppl:.2f} -> ~6.43)')
print(f'  - INT4 bit-packing for codes (VRAM {vram:.1f} GB -> ~7 GB)')
print(f'  - torch.compile(model) for remaining Python overhead')